In [1]:
# ==========================================================
# FINAL CORRECT CODE — TRUE ViT3D SEED42
# CPU SAFE + STEP PRINTS + FAST MRI FILTER
#
# Goal:
#   1) Build labels exactly like your Fed Learning code
#   2) Link labels with MRI folders by PATNO
#   3) Train ViT3D one time with seed 42
#   4) Split patients 70/15/15 train/val/test
#   5) Save:
#        - vit3d_multitask_trained_seed42.pt
#        - vit3d_extractor_trained_seed42.pt
#        - mri_features.csv
#
# IMPORTANT:
#   - -1 means missing label and is ignored in the loss.
#   - ViT3D trains on CPU to avoid Kaggle CUDA error.
#   - For a quick test, keep EPOCHS = 1.
# ==========================================================

import re
import gc
import random
import warnings
import shutil
from pathlib import Path
from functools import reduce

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import pydicom

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from scipy.ndimage import zoom, gaussian_filter, binary_erosion, binary_dilation, binary_fill_holes

try:
    from tqdm.auto import tqdm
    HAS_TQDM = True
except Exception:
    HAS_TQDM = False

warnings.filterwarnings("ignore")

# ==========================================================
# 0) CONFIG
# ==========================================================
SEED = 42

IMAGE_BASE = Path("/kaggle/input/datasets/ANONYMIZED_USER/dataimage/dataimage/imagedataset/image")
KAGGLE_INPUT = Path("/kaggle/input")

OUT_ROOT = Path("/kaggle/working/TRUE_VIT3D_SEED42_CPU_FINAL")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

MRI_TARGET_SHAPE = (96, 96, 96)
MRI_PATCH_SIZE = (16, 16, 16)
MRI_EMBED_DIM = 128
MRI_DEPTH = 4
MRI_NUM_HEADS = 4
MRI_FEAT_DIM = 256

EPOCHS = 15         # final run
BATCH_SIZE = 1
LR = 1e-4
WEIGHT_DECAY = 1e-4

device = torch.device("cpu")
print("Device forced:", device)
try:
    torch.set_num_threads(2)
except Exception:
    pass

# ==========================================================
# 1) BASIC HELPERS
# ==========================================================
KEYS = ["PATNO", "EVENT_ID"]
DATE_CANDIDATES = ["INFODT", "VISITDT", "DATE", "EXAMDATE", "TESTDATE"]

ADMIN_WORDS = [
    "REC_ID", "PAG_NAME", "ORIG_ENTRY", "LAST_UPDATE",
    "ENTRY", "UPDATE", "NUPSOURC", "PTCGBOTH",
    "INFODT", "VISITDT", "EXAMDATE", "TESTDATE", "DATE"
]

LEAKY_CONTAINS = [
    "ESS_TOTAL", "SCORE_GDS", "SCAUTOT", "RBDTOT",
    "NP1RTOT", "NP2PTOT", "NP3TOT", "NHY"
]

_FR_MONTH = {
    "janv": "Jan", "févr": "Feb", "fevr": "Feb", "mars": "Mar", "avr": "Apr",
    "mai": "May", "juin": "Jun", "juil": "Jul", "août": "Aug", "aout": "Aug",
    "sept": "Sep", "oct": "Oct", "nov": "Nov", "déc": "Dec", "dec": "Dec"
}

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(SEED)

def find_file(filename):
    matches = list(KAGGLE_INPUT.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"{filename} introuvable dans /kaggle/input")
    print(f"[FOUND] {filename} -> {matches[0]}")
    return matches[0]

def read_csv_robust(path):
    return pd.read_csv(path, encoding="latin1", sep=None, engine="python", on_bad_lines="skip")

def norm_cols(df):
    df = df.copy()
    df.columns = [str(c).strip().upper() for c in df.columns]
    return df

def coerce_keys(df):
    df = df.copy()
    if "PATNO" in df.columns:
        df["PATNO"] = pd.to_numeric(df["PATNO"], errors="coerce").astype("Int64")
    if "EVENT_ID" in df.columns:
        df["EVENT_ID"] = df["EVENT_ID"].astype(str).str.upper().str.strip()
    return df

def detect_date_col(df):
    for c in DATE_CANDIDATES:
        if c in df.columns:
            return c
    return None

def parse_date_series(s):
    s = s.astype(str).str.strip()
    for fr, en in _FR_MONTH.items():
        s = s.str.replace(fr, en, regex=False)
    return pd.to_datetime(s, errors="coerce", dayfirst=True)

def dedup_by_key_keep_latest(df, keys=None):
    df = df.copy()
    if keys is None:
        keys = [k for k in ["PATNO", "EVENT_ID"] if k in df.columns]
    if len(keys) == 0:
        return df.drop_duplicates(keep="last")

    dcol = detect_date_col(df)
    if dcol is not None:
        df["_DATE_TMP_"] = parse_date_series(df[dcol])
        df = df.sort_values(keys + ["_DATE_TMP_"])
        df = df.drop_duplicates(keys, keep="last")
        df = df.drop(columns=["_DATE_TMP_"])
    else:
        df = df.drop_duplicates(keys, keep="last")
    return df

def prefix_features(df, prefix, protected=("PATNO", "EVENT_ID")):
    df = df.copy()
    rename_map = {}
    for c in df.columns:
        if c not in protected:
            rename_map[c] = f"{prefix}{c}"
    return df.rename(columns=rename_map)

def event_id_to_month(x):
    x = str(x).strip().upper()
    if x == "BL":
        return 0
    m = re.fullmatch(r"V(\d+)", x)
    if m:
        v = int(m.group(1))
        return (v - 2) * 6
    return np.nan

def drop_admin_cols_safe(df):
    drop_cols = []
    for c in df.columns:
        cu = c.upper()
        if cu in KEYS:
            continue
        if any(w in cu for w in ADMIN_WORDS):
            drop_cols.append(c)
    return df.drop(columns=drop_cols, errors="ignore")

def drop_leaky_cols_safe(df):
    drop_cols = []
    for c in df.columns:
        cu = c.upper()
        if cu in KEYS:
            continue
        if any(w in cu for w in LEAKY_CONTAINS):
            drop_cols.append(c)
    return df.drop(columns=drop_cols, errors="ignore")

def filter_to_pd(participant_status):
    df = participant_status.copy()
    candidate_cols = [c for c in df.columns if ("COHORT" in c or "STATUS" in c or "DEF" in c)]
    if len(candidate_cols) == 0:
        txt = df.astype(str).agg(" ".join, axis=1)
    else:
        txt = pd.Series("", index=df.index)
        for c in candidate_cols:
            txt = txt.astype(str) + " " + df[c].astype(str)
    mask = txt.str.contains("PARKINSON", case=False, na=False)
    keep_pat = df.loc[mask, "PATNO"].dropna().astype(int).unique()
    return set(keep_pat)

# ==========================================================
# 2) MRI FOLDER MAP
# ==========================================================
def build_image_patient_folder_map(image_base: Path):
    mp = {}
    for p in image_base.iterdir():
        if p.is_dir() and p.name.strip().isdigit():
            mp[int(p.name.strip())] = p
    print("MRI patient folders found:", len(mp))
    return mp

IMAGE_PATIENT_FOLDER_MAP = build_image_patient_folder_map(IMAGE_BASE)

def resolve_patient_folder(patient_id):
    pid = int(patient_id)
    return IMAGE_PATIENT_FOLDER_MAP.get(pid, None)

def get_mri_patient_ids(image_base: Path):
    return sorted(list(IMAGE_PATIENT_FOLDER_MAP.keys()))

# ==========================================================
# 3) LOAD CLINICAL LIKE FED
# ==========================================================
def load_all_clinical_like_fed():
    print("STEP 1: loading clinical files...")

    demo_path   = find_file("Demographics.csv")
    ess_path    = find_file("Epworth_Sleepiness_Scale.csv")
    up1_path    = find_file("MDS-UPDRS_Part_I.csv")
    up4_path    = find_file("MDS-UPDRS_Part_IV.csv")
    rem_path    = find_file("REM_Sleep_Behavior.csv")
    scopa_path  = find_file("SCOPA-AUT.csv")
    gds_path    = find_file("Geriatric_Depression_Scale.csv")
    status_path = find_file("Participant_Status.csv")
    up23_path   = find_file("UPDRS_PartII_PartIII.xlsx")

    demo = prefix_features(dedup_by_key_keep_latest(coerce_keys(norm_cols(read_csv_robust(demo_path))), keys=["PATNO", "EVENT_ID"]), "DEMO__")
    ess = prefix_features(dedup_by_key_keep_latest(coerce_keys(norm_cols(read_csv_robust(ess_path))), keys=["PATNO", "EVENT_ID"]), "ESS__")
    up1 = prefix_features(dedup_by_key_keep_latest(coerce_keys(norm_cols(read_csv_robust(up1_path))), keys=["PATNO", "EVENT_ID"]), "UP1__")
    up4 = prefix_features(dedup_by_key_keep_latest(coerce_keys(norm_cols(read_csv_robust(up4_path))), keys=["PATNO", "EVENT_ID"]), "UP4__")
    rem = prefix_features(dedup_by_key_keep_latest(coerce_keys(norm_cols(read_csv_robust(rem_path))), keys=["PATNO", "EVENT_ID"]), "REM__")
    scopa = prefix_features(dedup_by_key_keep_latest(coerce_keys(norm_cols(read_csv_robust(scopa_path))), keys=["PATNO", "EVENT_ID"]), "SCOPA__")
    gds = prefix_features(dedup_by_key_keep_latest(coerce_keys(norm_cols(read_csv_robust(gds_path))), keys=["PATNO", "EVENT_ID"]), "GDS__")

    status = dedup_by_key_keep_latest(coerce_keys(norm_cols(read_csv_robust(status_path))), keys=["PATNO"])

    up23 = pd.read_excel(up23_path)
    up23 = prefix_features(dedup_by_key_keep_latest(coerce_keys(norm_cols(up23)), keys=["PATNO", "EVENT_ID"]), "UP23__")

    demo  = drop_leaky_cols_safe(drop_admin_cols_safe(demo))
    ess   = drop_leaky_cols_safe(drop_admin_cols_safe(ess))
    up1   = drop_leaky_cols_safe(drop_admin_cols_safe(up1))
    up4   = drop_leaky_cols_safe(drop_admin_cols_safe(up4))
    rem   = drop_leaky_cols_safe(drop_admin_cols_safe(rem))
    scopa = drop_leaky_cols_safe(drop_admin_cols_safe(scopa))
    gds   = drop_leaky_cols_safe(drop_admin_cols_safe(gds))
    up23  = drop_leaky_cols_safe(drop_admin_cols_safe(up23))

    dfs = [demo, ess, up1, up4, rem, scopa, gds, up23]
    df_seq = reduce(lambda left, right: pd.merge(left, right, on=KEYS, how="outer"), dfs)

    status = coerce_keys(norm_cols(status))
    pd_patients = filter_to_pd(status)
    df_seq = df_seq[df_seq["PATNO"].isin(pd_patients)].copy()

    df_seq["MONTH"] = df_seq["EVENT_ID"].apply(event_id_to_month)
    df_seq = df_seq[df_seq["MONTH"].notna()].copy()
    df_seq["MONTH"] = df_seq["MONTH"].astype(int)
    df_seq = df_seq.sort_values(["PATNO", "MONTH"])
    df_seq = df_seq.drop_duplicates(subset=["PATNO", "MONTH"], keep="last").copy()

    print("Merged clinical shape:", df_seq.shape)
    return df_seq

def build_labels_same_as_fed(df_seq):
    print("STEP 2: building labels exactly like Fed...")
    df_seq = df_seq.copy()

    if "GDS__DEPRESSION" not in df_seq.columns:
        gds_score_candidates = [c for c in df_seq.columns if "GDS" in c and ("TOTAL" in c or "SCORE" in c)]
        if len(gds_score_candidates) == 0:
            raise ValueError("No GDS score column found.")
        score = pd.to_numeric(df_seq[gds_score_candidates[0]], errors="coerce")
        df_seq["GDS__DEPRESSION"] = (score >= 5).astype("Float64")

    if "GDS__LEVELS" not in df_seq.columns:
        gds_score_candidates = [c for c in df_seq.columns if "GDS" in c and ("TOTAL" in c or "SCORE" in c)]
        if len(gds_score_candidates) == 0:
            raise ValueError("No GDS score column found.")
        score = pd.to_numeric(df_seq[gds_score_candidates[0]], errors="coerce")

        levels = pd.Series(np.nan, index=df_seq.index)
        levels[(score >= 0) & (score <= 4)] = 0
        levels[(score >= 5) & (score <= 8)] = 1
        levels[(score >= 9) & (score <= 11)] = 2
        levels[(score >= 12)] = 3
        df_seq["GDS__LEVELS"] = levels.astype("Float64")

    if "SUBTYPE_BIN" not in df_seq.columns:
        ratio_candidates = [c for c in df_seq.columns if "RATIO" in c.upper()]
        td_candidates = [c for c in df_seq.columns if "SCORE TD" in c.upper()]
        pigd_candidates = [c for c in df_seq.columns if "SCORE PIGD" in c.upper()]
        subtype = pd.Series(np.nan, index=df_seq.index)

        if len(ratio_candidates) > 0:
            ratio = pd.to_numeric(df_seq[ratio_candidates[0]], errors="coerce")
            subtype[ratio >= 1.15] = 1
            subtype[ratio <= 0.90] = 2
        elif len(td_candidates) > 0 and len(pigd_candidates) > 0:
            td = pd.to_numeric(df_seq[td_candidates[0]], errors="coerce")
            pigd = pd.to_numeric(df_seq[pigd_candidates[0]], errors="coerce")
            ratio = td / pigd.replace(0, np.nan)
            subtype[ratio >= 1.15] = 1
            subtype[ratio <= 0.90] = 2
        else:
            raise ValueError("Impossible de construire SUBTYPE_BIN")

        df_seq["SUBTYPE_BIN"] = subtype.astype("Float64")

    print("Labels ready exactly like Fed.")
    return df_seq

def make_patient_labels_from_fed_logic(df_seq, last_month=12):
    print("STEP 3: creating patient label table...")
    rows = []

    for patno, g in df_seq.groupby("PATNO"):
        g = g.copy().sort_values("MONTH").drop_duplicates(subset=["MONTH"], keep="last").set_index("MONTH")
        if last_month not in g.index:
            continue

        row_last = g.loc[last_month]
        if isinstance(row_last, pd.DataFrame):
            row_last = row_last.iloc[-1]

        y_subtype = row_last["SUBTYPE_BIN"] if "SUBTYPE_BIN" in row_last.index else pd.NA
        y_depbin  = row_last["GDS__DEPRESSION"] if "GDS__DEPRESSION" in row_last.index else pd.NA
        y_levels  = row_last["GDS__LEVELS"] if "GDS__LEVELS" in row_last.index else pd.NA

        if pd.isna(y_subtype) and pd.isna(y_depbin) and pd.isna(y_levels):
            continue

        if pd.isna(y_subtype):
            ys = -1
        else:
            ys = int(y_subtype)
            if ys in [1, 2]:
                ys -= 1
            if ys not in [0, 1]:
                ys = -1

        if pd.isna(y_depbin):
            yd = -1
        else:
            yd = int(y_depbin)
            if yd not in [0, 1]:
                yd = -1

        if pd.isna(y_levels):
            yl = -1
        else:
            yl = int(y_levels)
            if yl not in [0, 1, 2, 3]:
                yl = -1

        rows.append({"PATNO": int(patno), "y_subtype": ys, "y_dep": yd, "y_level": yl})

    labels = pd.DataFrame(rows).drop_duplicates("PATNO").reset_index(drop=True)

    print("Patient labels from Fed logic:", labels.shape)
    print("Subtype (-1 ignored):", labels["y_subtype"].value_counts().sort_index())
    print("Dep (-1 ignored):", labels["y_dep"].value_counts().sort_index())
    print("Levels (-1 ignored):", labels["y_level"].value_counts().sort_index())
    return labels

def build_mri_label_df_same_as_fed():
    df_seq = load_all_clinical_like_fed()
    df_seq = build_labels_same_as_fed(df_seq)
    labels = make_patient_labels_from_fed_logic(df_seq, last_month=12)

    print("STEP 4: fast MRI folder filtering...")
    valid_rows = []
    skip_no_mri = 0
    skip_bad_mri = 0
    skip_no_valid_label = 0

    iterator = labels.iterrows()
    if HAS_TQDM:
        iterator = tqdm(iterator, total=len(labels), desc="Filtering MRI folders")

    for idx, (_, r) in enumerate(iterator, start=1):
        if (not HAS_TQDM) and (idx % 50 == 0 or idx == len(labels)):
            print(f"Filtering MRI folders: {idx}/{len(labels)}")

        pid = int(r["PATNO"])

        if int(r["y_subtype"]) < 0 and int(r["y_dep"]) < 0 and int(r["y_level"]) < 0:
            skip_no_valid_label += 1
            continue

        folder = resolve_patient_folder(pid)
        if folder is None:
            skip_no_mri += 1
            continue

        dcm_files = list(Path(folder).rglob("*.dcm"))
        if len(dcm_files) < 2:
            skip_bad_mri += 1
            continue

        valid_rows.append(r)

    df = pd.DataFrame(valid_rows).drop_duplicates("PATNO").reset_index(drop=True)

    if df.empty:
        raise RuntimeError("No patients with Fed labels + MRI folder.")

    print("Patients with Fed labels + MRI folder:", df.shape)
    print("Skipped no valid label:", skip_no_valid_label)
    print("Skipped no MRI folder:", skip_no_mri)
    print("Skipped bad MRI:", skip_bad_mri)
    print("Subtype (-1 ignored):", df["y_subtype"].value_counts().sort_index())
    print("Dep (-1 ignored):", df["y_dep"].value_counts().sort_index())
    print("Level (-1 ignored):", df["y_level"].value_counts().sort_index())
    return df

# ==========================================================
# 4) MRI PREPROCESSING
# ==========================================================
def read_dicom_series(folder):
    dcm_files = list(Path(folder).rglob("*.dcm"))
    if len(dcm_files) == 0:
        return None

    slices = []
    target_shape = None

    for f in dcm_files:
        try:
            ds = pydicom.dcmread(str(f), force=True)
            arr = ds.pixel_array.astype(np.float32)

            if arr.ndim == 3:
                arr = arr[..., 0]
            if arr.ndim != 2:
                continue

            if target_shape is None:
                target_shape = arr.shape
            if arr.shape != target_shape:
                continue

            z_pos = None
            if hasattr(ds, "ImagePositionPatient"):
                try:
                    z_pos = float(ds.ImagePositionPatient[2])
                except Exception:
                    z_pos = None
            if z_pos is None and hasattr(ds, "SliceLocation"):
                try:
                    z_pos = float(ds.SliceLocation)
                except Exception:
                    z_pos = None
            if z_pos is None:
                z_pos = len(slices)

            slices.append((z_pos, arr))
        except Exception:
            continue

    if len(slices) < 2:
        return None

    slices = sorted(slices, key=lambda x: x[0])
    vol = np.stack([s[1] for s in slices], axis=-1).astype(np.float32)
    return vol

def reorient_to_standard(vol):
    vol = np.asarray(vol, dtype=np.float32)
    if vol.ndim == 2:
        vol = np.expand_dims(vol, axis=-1)
    if vol.ndim == 4:
        vol = vol[..., 0]
    if vol.ndim != 3:
        raise ValueError(f"MRI volume must be 3D, got shape={vol.shape}")
    if vol.shape[0] < vol.shape[-1]:
        vol = np.transpose(vol, (2, 0, 1))
    if vol.shape[-1] > 1:
        vol = vol[..., ::-1]
    return vol.astype(np.float32)

def bias_field_correction(vol):
    vol = np.asarray(vol, dtype=np.float32)
    smooth = gaussian_filter(vol, sigma=3)
    corrected = vol / (smooth + 1e-6)
    corrected = corrected - corrected.min()
    if corrected.max() > 0:
        corrected = corrected / corrected.max()
    return corrected.astype(np.float32)

def skull_stripping(vol):
    vol = np.asarray(vol, dtype=np.float32)
    nonzero = vol[vol > 0]
    if nonzero.size == 0:
        return vol.astype(np.float32)
    thr = np.percentile(nonzero, 35)
    mask = vol > thr
    mask = binary_erosion(mask, iterations=1)
    mask = binary_dilation(mask, iterations=2)
    mask = binary_fill_holes(mask)
    return (vol * mask.astype(np.float32)).astype(np.float32)

def register_to_mni(vol, out_shape=(96, 96, 96)):
    # Faster CPU version: direct resize to target shape.
    vol = np.asarray(vol, dtype=np.float32)
    factors = [o / s for o, s in zip(out_shape, vol.shape)]
    return zoom(vol, zoom=factors, order=1).astype(np.float32)

def center_crop_or_pad(vol, target_shape=(96, 96, 96)):
    vol = np.asarray(vol, dtype=np.float32)
    pads = []
    for s, t in zip(vol.shape, target_shape):
        if s < t:
            total = t - s
            left = total // 2
            right = total - left
            pads.append((left, right))
        else:
            pads.append((0, 0))

    result = np.pad(vol, pads, mode="constant")
    slices = []
    for s, t in zip(result.shape, target_shape):
        start = max((s - t) // 2, 0)
        end = start + t
        slices.append(slice(start, end))

    return result[slices[0], slices[1], slices[2]].astype(np.float32)

def normalize_final_volume(vol):
    vol = np.asarray(vol, dtype=np.float32)
    if np.std(vol) > 1e-8:
        vol = (vol - np.mean(vol)) / (np.std(vol) + 1e-8)
    p1, p99 = np.percentile(vol, 1), np.percentile(vol, 99)
    vol = np.clip(vol, p1, p99)
    vol = vol - vol.min()
    if vol.max() > 0:
        vol = vol / vol.max()
    return vol.astype(np.float32)

def preprocess_mri_volume(vol):
    vol = reorient_to_standard(vol)
    vol = bias_field_correction(vol)
    vol = skull_stripping(vol)
    vol = register_to_mni(vol, out_shape=MRI_TARGET_SHAPE)
    vol = center_crop_or_pad(vol, target_shape=MRI_TARGET_SHAPE)
    vol = normalize_final_volume(vol)
    return vol.astype(np.float32)

def load_mri_for_patient(patient_id):
    folder = resolve_patient_folder(patient_id)
    if folder is None:
        return None
    vol = read_dicom_series(folder)
    if vol is None:
        return None
    try:
        return preprocess_mri_volume(vol)
    except Exception as e:
        print(f"[WARN] MRI preprocessing failed for {patient_id}: {e}")
        return None

# ==========================================================
# 5) MODEL
# ==========================================================
class PatchEmbed3D(nn.Module):
    def __init__(self, patch_size=(16, 16, 16), in_chans=1, embed_dim=128):
        super().__init__()
        self.proj = nn.Conv3d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x

class ViT3D_Extractor(nn.Module):
    def __init__(self):
        super().__init__()

        self.patch_embed = PatchEmbed3D(
            patch_size=MRI_PATCH_SIZE,
            in_chans=1,
            embed_dim=MRI_EMBED_DIM
        )

        num_patches = (
            (MRI_TARGET_SHAPE[0] // MRI_PATCH_SIZE[0]) *
            (MRI_TARGET_SHAPE[1] // MRI_PATCH_SIZE[1]) *
            (MRI_TARGET_SHAPE[2] // MRI_PATCH_SIZE[2])
        )

        self.cls_token = nn.Parameter(torch.zeros(1, 1, MRI_EMBED_DIM))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, MRI_EMBED_DIM))
        self.pos_drop = nn.Dropout(0.10)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=MRI_EMBED_DIM,
            nhead=MRI_NUM_HEADS,
            dim_feedforward=int(MRI_EMBED_DIM * 4),
            dropout=0.10,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=MRI_DEPTH)
        self.norm = nn.LayerNorm(MRI_EMBED_DIM)

        self.head = nn.Sequential(
            nn.Linear(MRI_EMBED_DIM * 2, 512),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(512, MRI_FEAT_DIM)
        )

    def forward(self, x):
        x = self.patch_embed(x)
        B = x.shape[0]

        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)

        x = x + self.pos_embed[:, :x.size(1), :]
        x = self.pos_drop(x)

        x = self.encoder(x)
        x = self.norm(x)

        cls_feat = x[:, 0]
        mean_feat = x[:, 1:].mean(dim=1)

        return self.head(torch.cat([cls_feat, mean_feat], dim=1))

class ViT3D_Multitask(nn.Module):
    def __init__(self):
        super().__init__()
        self.extractor = ViT3D_Extractor()
        self.subtype_head = nn.Linear(MRI_FEAT_DIM, 2)
        self.dep_head = nn.Linear(MRI_FEAT_DIM, 2)
        self.level_head = nn.Linear(MRI_FEAT_DIM, 4)

    def forward(self, x):
        feat = self.extractor(x)
        return {
            "feat": feat,
            "subtype": self.subtype_head(feat),
            "dep": self.dep_head(feat),
            "level": self.level_head(feat)
        }

# ==========================================================
# 6) DATASET + LOSS
# ==========================================================
class MRIMultitaskDataset(Dataset):
    def __init__(self, df, cache_volumes=False):
        self.df = df.reset_index(drop=True)
        self.cache_volumes = cache_volumes
        self.cache = {}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        pid = int(r["PATNO"])

        if self.cache_volumes and pid in self.cache:
            vol = self.cache[pid]
        else:
            vol = load_mri_for_patient(pid)
            if vol is None:
                # Return zeros instead of crashing.
                vol = np.zeros(MRI_TARGET_SHAPE, dtype=np.float32)
            if self.cache_volumes:
                self.cache[pid] = vol

        x = torch.tensor(vol, dtype=torch.float32).unsqueeze(0)

        return {
            "x": x,
            "y_subtype": torch.tensor(int(r["y_subtype"]), dtype=torch.long),
            "y_dep": torch.tensor(int(r["y_dep"]), dtype=torch.long),
            "y_level": torch.tensor(int(r["y_level"]), dtype=torch.long),
            "PATNO": pid
        }

def multitask_vit_loss(outputs, batch):
    loss = 0.0
    n = 0

    y = batch["y_subtype"].to(device)
    mask = y >= 0
    if mask.any():
        loss = loss + F.cross_entropy(outputs["subtype"][mask], y[mask])
        n += 1

    y = batch["y_dep"].to(device)
    mask = y >= 0
    if mask.any():
        loss = loss + F.cross_entropy(outputs["dep"][mask], y[mask])
        n += 1

    y = batch["y_level"].to(device)
    mask = y >= 0
    if mask.any():
        loss = loss + F.cross_entropy(outputs["level"][mask], y[mask])
        n += 1

    if n == 0:
        return None

    return loss / n

# ==========================================================
# 7) TRAIN + EXTRACT
# ==========================================================
def train_multitask_vit3d_seed42(label_df):
    print("STEP 5: training ViT3D...")
    set_seed(SEED)

    seed_dir = OUT_ROOT / f"seed_{SEED}"
    seed_dir.mkdir(parents=True, exist_ok=True)

    # Patient-wise split: 70% train, 15% validation, 15% test
    train_df, temp_df = train_test_split(
        label_df,
        test_size=0.30,
        random_state=SEED,
        stratify=None
    )

    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=SEED,
        stratify=None
    )

    print("Train patients:", len(train_df), "| Val patients:", len(val_df), "| Test patients:", len(test_df))

    split_dir = seed_dir / "split_70_15_15"
    split_dir.mkdir(parents=True, exist_ok=True)
    train_df.to_csv(split_dir / "train_patients.csv", index=False)
    val_df.to_csv(split_dir / "val_patients.csv", index=False)
    test_df.to_csv(split_dir / "test_patients.csv", index=False)

    train_loader = DataLoader(MRIMultitaskDataset(train_df), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(MRIMultitaskDataset(val_df), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader = DataLoader(MRIMultitaskDataset(test_df), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = ViT3D_Multitask().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_val_loss = float("inf")
    best_state = None

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_losses = []

        iterator = train_loader
        if HAS_TQDM:
            iterator = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{EPOCHS} train", leave=False)

        for step, batch in enumerate(iterator, start=1):
            if (not HAS_TQDM) and (step % 10 == 0 or step == len(train_loader)):
                print(f"Epoch {epoch}/{EPOCHS} train batch {step}/{len(train_loader)}")

            x = batch["x"].to(device)

            optimizer.zero_grad(set_to_none=True)
            outputs = model(x)
            loss = multitask_vit_loss(outputs, batch)

            if loss is None:
                continue

            loss.backward()
            optimizer.step()
            train_losses.append(float(loss.item()))

            del x, outputs, loss
            gc.collect()

        model.eval()
        val_losses = []
        correct = {"subtype": 0, "dep": 0, "level": 0}
        total = {"subtype": 0, "dep": 0, "level": 0}

        with torch.no_grad():
            iterator = val_loader
            if HAS_TQDM:
                iterator = tqdm(val_loader, desc=f"Epoch {epoch:02d}/{EPOCHS} val", leave=False)

            for batch in iterator:
                x = batch["x"].to(device)
                outputs = model(x)
                loss = multitask_vit_loss(outputs, batch)

                if loss is not None:
                    val_losses.append(float(loss.item()))

                for task, ykey in [("subtype", "y_subtype"), ("dep", "y_dep"), ("level", "y_level")]:
                    y = batch[ykey].to(device)
                    mask = y >= 0
                    if mask.any():
                        pred = outputs[task][mask].argmax(dim=1)
                        correct[task] += int((pred == y[mask]).sum().item())
                        total[task] += int(mask.sum().item())

                del x, outputs, loss
                gc.collect()

        train_loss = float(np.mean(train_losses)) if train_losses else float("nan")
        val_loss = float(np.mean(val_losses)) if val_losses else float("inf")
        acc = {k: correct[k] / max(total[k], 1) for k in correct}

        print(
            f"[Seed {SEED}] Epoch {epoch:02d}/{EPOCHS} | "
            f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | acc={acc}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is None:
        raise RuntimeError("Training failed.")

    model.load_state_dict(best_state)

    # Final test evaluation
    model.eval()
    test_losses = []
    correct = {"subtype": 0, "dep": 0, "level": 0}
    total = {"subtype": 0, "dep": 0, "level": 0}

    with torch.no_grad():
        iterator = test_loader
        if HAS_TQDM:
            iterator = tqdm(test_loader, desc="Final test", leave=False)

        for batch in iterator:
            x = batch["x"].to(device)
            outputs = model(x)
            loss = multitask_vit_loss(outputs, batch)

            if loss is not None:
                test_losses.append(float(loss.item()))

            for task, ykey in [("subtype", "y_subtype"), ("dep", "y_dep"), ("level", "y_level")]:
                y = batch[ykey].to(device)
                mask = y >= 0
                if mask.any():
                    pred = outputs[task][mask].argmax(dim=1)
                    correct[task] += int((pred == y[mask]).sum().item())
                    total[task] += int(mask.sum().item())

    test_loss = float(np.mean(test_losses)) if test_losses else float("inf")
    test_acc = {k: correct[k] / max(total[k], 1) for k in correct}

    metrics = {
        "seed": SEED,
        "epochs": EPOCHS,
        "train_patients": int(len(train_df)),
        "val_patients": int(len(val_df)),
        "test_patients": int(len(test_df)),
        "best_val_loss": float(best_val_loss),
        "test_loss": float(test_loss),
        "test_acc_subtype": float(test_acc["subtype"]),
        "test_acc_dep": float(test_acc["dep"]),
        "test_acc_level": float(test_acc["level"]),
    }
    pd.DataFrame([metrics]).to_csv(seed_dir / "vit3d_test_metrics.csv", index=False)

    print("Final test loss:", test_loss)
    print("Final test acc:", test_acc)

    multitask_path = seed_dir / f"vit3d_multitask_trained_seed{SEED}.pt"
    extractor_path = seed_dir / f"vit3d_extractor_trained_seed{SEED}.pt"

    torch.save(model.state_dict(), multitask_path)
    torch.save(model.extractor.state_dict(), extractor_path)

    print("Saved full multitask ViT3D:", multitask_path)
    print("Saved trained ViT3D extractor:", extractor_path)
    print("Saved split files:", split_dir)
    print("Saved test metrics:", seed_dir / "vit3d_test_metrics.csv")

    return extractor_path

def extract_features_seed42(extractor_path):
    print("STEP 6: extracting MRI features...")
    seed_dir = OUT_ROOT / f"seed_{SEED}"

    extractor = ViT3D_Extractor().to(device)
    extractor.load_state_dict(torch.load(extractor_path, map_location=device))
    extractor.eval()

    patient_ids = get_mri_patient_ids(IMAGE_BASE)
    rows = []
    ok_count = 0
    miss_count = 0

    iterator = patient_ids
    if HAS_TQDM:
        iterator = tqdm(patient_ids, total=len(patient_ids), desc="Extracting MRI features")

    with torch.no_grad():
        for i, pat_id in enumerate(iterator, start=1):
            if (not HAS_TQDM) and (i % 20 == 0 or i == len(patient_ids)):
                print(f"Feature extraction: {i}/{len(patient_ids)}")

            vol = load_mri_for_patient(pat_id)

            if vol is None:
                miss_count += 1
                row = {"PATNO": int(pat_id), "mri_available": 0}
                for j in range(MRI_FEAT_DIM):
                    row[f"mri_feat_{j:03d}"] = 0.0
                rows.append(row)
                continue

            try:
                x = torch.tensor(vol, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
                feat = extractor(x).cpu().numpy().flatten().astype(np.float32)

                row = {"PATNO": int(pat_id), "mri_available": 1}
                for j in range(MRI_FEAT_DIM):
                    row[f"mri_feat_{j:03d}"] = float(feat[j])
                rows.append(row)
                ok_count += 1

                del x
                gc.collect()

            except Exception as e:
                print(f"[WARN] Extraction failed for patient {pat_id}: {e}")
                miss_count += 1
                row = {"PATNO": int(pat_id), "mri_available": 0}
                for j in range(MRI_FEAT_DIM):
                    row[f"mri_feat_{j:03d}"] = 0.0
                rows.append(row)

    df = pd.DataFrame(rows).sort_values("PATNO").reset_index(drop=True)

    final_csv = OUT_ROOT / "mri_features.csv"
    final_pkl = OUT_ROOT / "mri_features.pkl"
    xai_csv = OUT_ROOT / "mri_features_seed42_for_fed_and_xai.csv"

    df.to_csv(final_csv, index=False)
    df.to_pickle(final_pkl)
    df.to_csv(xai_csv, index=False)

    print("MRI features extracted:", ok_count)
    print("MRI missing/failed:", miss_count)
    print("Saved:", final_csv)
    print("Saved:", final_pkl)
    print("Saved:", xai_csv)
    print("Shape:", df.shape)
    print(df.head())

    return df


def collect_fed_artifacts_if_exist():
    """
    This ViT3D notebook cannot create Fed Learning artifacts.
    It can only copy them into the same output folder IF they already exist
    after you run the Fed Learning notebook.

    Expected Fed artifacts:
        global_model_complete.pt
        preprocessing.pkl
    """
    print("\nSTEP 7: searching Fed Learning artifacts...")

    names = ["global_model_complete.pt", "preprocessing.pkl"]
    search_roots = [Path("/kaggle/working"), Path("/kaggle/input")]

    copied = []
    missing = []

    for name in names:
        found = []
        for root in search_roots:
            if root.exists():
                found += list(root.rglob(name))

        # avoid copying file from OUT_ROOT to itself
        found = [p for p in found if OUT_ROOT not in p.parents]

        if len(found) == 0:
            print(f"[MISSING] {name} not found. Run Fed Learning notebook first.")
            missing.append(name)
            continue

        src = found[0]
        dst = OUT_ROOT / name
        shutil.copy2(src, dst)
        print(f"[COPIED] {src} -> {dst}")
        copied.append(str(dst))

    manifest = {
        "vit3d_extractor": str(OUT_ROOT / "seed_42" / "vit3d_extractor_trained_seed42.pt"),
        "vit3d_multitask": str(OUT_ROOT / "seed_42" / "vit3d_multitask_trained_seed42.pt"),
        "mri_features_csv": str(OUT_ROOT / "mri_features.csv"),
        "fed_global_model_complete": str(OUT_ROOT / "global_model_complete.pt") if (OUT_ROOT / "global_model_complete.pt").exists() else None,
        "preprocessing_pkl": str(OUT_ROOT / "preprocessing.pkl") if (OUT_ROOT / "preprocessing.pkl").exists() else None,
        "copied": copied,
        "missing": missing,
    }

    pd.DataFrame([manifest]).to_csv(OUT_ROOT / "xai_artifacts_manifest.csv", index=False)
    print("Saved manifest:", OUT_ROOT / "xai_artifacts_manifest.csv")


def main():
    label_df = build_mri_label_df_same_as_fed()
    extractor_path = train_multitask_vit3d_seed42(label_df)
    _ = extract_features_seed42(extractor_path)
    collect_fed_artifacts_if_exist()

    print("\nDONE.")
    print("Use this file in Fed Learning:")
    print(OUT_ROOT / "mri_features.csv")
    print("Use this trained extractor for Grad-CAM:")
    print(OUT_ROOT / "seed_42" / "vit3d_extractor_trained_seed42.pt")
    print("Fed artifacts copied here if already available:")
    print(OUT_ROOT / "global_model_complete.pt")
    print(OUT_ROOT / "preprocessing.pkl")

main()

Device forced: cpu
MRI patient folders found: 543
STEP 1: loading clinical files...
[FOUND] Demographics.csv -> /kaggle/input/datasets/ANONYMIZED_USER/dataset-maladie/dataset maladie/Demographics.csv
[FOUND] Epworth_Sleepiness_Scale.csv -> /kaggle/input/datasets/ANONYMIZED_USER/dataset-maladie/dataset maladie/Epworth_Sleepiness_Scale.csv
[FOUND] MDS-UPDRS_Part_I.csv -> /kaggle/input/datasets/ANONYMIZED_USER/dataset-maladie/dataset maladie/MDS-UPDRS_Part_I.csv
[FOUND] MDS-UPDRS_Part_IV.csv -> /kaggle/input/datasets/ANONYMIZED_USER/dataset-maladie/dataset maladie/MDS-UPDRS_Part_IV.csv
[FOUND] REM_Sleep_Behavior.csv -> /kaggle/input/datasets/ANONYMIZED_USER/dataset-maladie/dataset maladie/REM_Sleep_Behavior.csv
[FOUND] SCOPA-AUT.csv -> /kaggle/input/datasets/ANONYMIZED_USER/dataset-maladie/dataset maladie/SCOPA-AUT.csv
[FOUND] Geriatric_Depression_Scale.csv -> /kaggle/input/datasets/ANONYMIZED_USER/dataset-maladie/dataset maladie/Geriatric_Depression_Scale.csv
[FOUND] Participant_Status.c

Filtering MRI folders:   0%|          | 0/1266 [00:00<?, ?it/s]

Patients with Fed labels + MRI folder: (455, 4)
Skipped no valid label: 0
Skipped no MRI folder: 769
Skipped bad MRI: 42
Subtype (-1 ignored): y_subtype
-1     48
 0    269
 1    138
Name: count, dtype: int64
Dep (-1 ignored): y_dep
-1      3
 0    100
 1    352
Name: count, dtype: int64
Level (-1 ignored): y_level
-1      3
 0    100
 1    346
 2      6
Name: count, dtype: int64
STEP 5: training ViT3D...
Train patients: 318 | Val patients: 68 | Test patients: 69


Epoch 01/15 train:   0%|          | 0/318 [00:00<?, ?it/s]

Epoch 01/15 val:   0%|          | 0/68 [00:00<?, ?it/s]

...

[Seed 42] Epoch 15/15 | train_loss=0.5333 | val_loss=0.7105 | acc={'subtype': 0.75, 'dep': 0.75, 'level': 0.7352941176470589}


Final test:   0%|          | 0/69 [00:00<?, ?it/s]

Final test loss: 0.631217851371005
Final test acc: {'subtype': 0.6666666666666666, 'dep': 0.7352941176470589, 'level': 0.7352941176470589}
Saved full multitask ViT3D: /kaggle/working/TRUE_VIT3D_SEED42_CPU_FINAL/seed_42/vit3d_multitask_trained_seed42.pt
Saved trained ViT3D extractor: /kaggle/working/TRUE_VIT3D_SEED42_CPU_FINAL/seed_42/vit3d_extractor_trained_seed42.pt
Saved split files: /kaggle/working/TRUE_VIT3D_SEED42_CPU_FINAL/seed_42/split_70_15_15
Saved test metrics: /kaggle/working/TRUE_VIT3D_SEED42_CPU_FINAL/seed_42/vit3d_test_metrics.csv
STEP 6: extracting MRI features...


Extracting MRI features:   0%|          | 0/543 [00:00<?, ?it/s]

MRI features extracted: 500
MRI missing/failed: 43
Saved: /kaggle/working/TRUE_VIT3D_SEED42_CPU_FINAL/mri_features.csv
Saved: /kaggle/working/TRUE_VIT3D_SEED42_CPU_FINAL/mri_features.pkl
Saved: /kaggle/working/TRUE_VIT3D_SEED42_CPU_FINAL/mri_features_seed42_for_fed_and_xai.csv
Shape: (543, 258)
   PATNO  mri_available  mri_feat_000  mri_feat_001  mri_feat_002  \
0   3028              1     -0.343689      0.384194     -0.051578   
1   3051              1     -0.310143      0.306762     -0.000721   
2   3052              1     -0.360304      0.368873     -0.058055   
3   3054              1     -0.316313      0.307198      0.004742   
4   3056              1     -0.338924      0.377632     -0.070056   

   mri_feat_003  mri_feat_004  mri_feat_005  mri_feat_006  mri_feat_007  ...  \
0     -0.543872     -0.322349      0.117832     -0.431790      0.719088  ...   
1     -0.267387     -0.183915      0.151279     -0.331961      0.561507  ...   
2     -0.478285     -0.323627      0.087006     -